## Develop a sequence-to-sequence model for machine translation

# 1) Imports

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import random
import os

# For reproducibility
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

# 2) Small toy dataset (English -> French)
# Replace with real dataset for production 

In [2]:
pairs = [
    ("hello", "bonjour"),
    ("how are you", "comment ça va"),
    ("i am fine", "je vais bien"),
    ("thank you", "merci"),
    ("good morning", "bonjour"),
    ("good night", "bonne nuit"),
    ("see you later", "à plus tard"),
    ("what is your name", "comment tu t'appelles"),
    ("my name is mamata", "je m'appelle mamata"),
    ("i love machine learning", "j'aime l'apprentissage automatique"),
    ("do you speak english", "parlez-vous anglais"),
    ("where is the restroom", "où sont les toilettes"),
    ("i need help", "j'ai besoin d'aide"),
    ("i am learning french", "j'apprends le français"),
    ("this is awesome", "c'est génial"),
]

# Shuffle and split
random.shuffle(pairs)
train_pairs = pairs[:12]
val_pairs = pairs[12:13]
test_pairs = pairs[13:]

# 3) Preprocess: tokenizers, sequences, padding
# We will add start and end tokens for target language

In [3]:
# Source (English) tokenizer
src_texts = [p[0] for p in train_pairs]
src_tokenizer = Tokenizer(filters='', lower=True, oov_token='<oov>')
src_tokenizer.fit_on_texts(src_texts)
src_vocab_size = len(src_tokenizer.word_index) + 1

# Target (French) tokenizer
tgt_texts = [p[1] for p in train_pairs]
# We'll add special tokens '<start>' and '<end>' to targets
tgt_texts_with_tokens = ['<start> ' + t + ' <end>' for t in tgt_texts]
tgt_tokenizer = Tokenizer(filters='', lower=True, oov_token='<oov>')
tgt_tokenizer.fit_on_texts(tgt_texts_with_tokens)
tgt_vocab_size = len(tgt_tokenizer.word_index) + 1

# Convert texts to sequences
def prepare_data(pairs, src_tok, tgt_tok, max_src_len=None, max_tgt_len=None):
    src_texts = [p[0] for p in pairs]
    tgt_texts = ['<start> ' + p[1] + ' <end>' for p in pairs]
    src_seq = src_tok.texts_to_sequences(src_texts)
    tgt_seq = tgt_tok.texts_to_sequences(tgt_texts)
    if max_src_len is None:
        max_src_len = max(len(s) for s in src_seq)
    if max_tgt_len is None:
        max_tgt_len = max(len(t) for t in tgt_seq)
    src_seq = pad_sequences(src_seq, maxlen=max_src_len, padding='post')
    tgt_seq = pad_sequences(tgt_seq, maxlen=max_tgt_len, padding='post')
    return src_seq, tgt_seq, max_src_len, max_tgt_len

X_train, y_train, max_src_len, max_tgt_len = prepare_data(train_pairs, src_tokenizer, tgt_tokenizer)
X_val, y_val, _, _ = prepare_data(val_pairs, src_tokenizer, tgt_tokenizer, max_src_len, max_tgt_len)
X_test, y_test, _, _ = prepare_data(test_pairs, src_tokenizer, tgt_tokenizer, max_src_len, max_tgt_len)

print("src_vocab_size:", src_vocab_size)
print("tgt_vocab_size:", tgt_vocab_size)
print("max_src_len:", max_src_len, "max_tgt_len:", max_tgt_len)
print("Example source seq:", X_train[0])
print("Example target seq:", y_train[0])

# Create decoder input (target input) and target output (shifted by 1)
# For training: decoder_input = <start> ... , decoder_target = ... <end>
decoder_input_train = y_train[:, :-1]
decoder_target_train = y_train[:, 1:]
decoder_input_val = y_val[:, :-1]
decoder_target_val = y_val[:, 1:]


src_vocab_size: 29
tgt_vocab_size: 34
max_src_len: 4 max_tgt_len: 6
Example source seq: [ 9  4  2 10]
Example target seq: [2 4 5 6 3 0]


# 4) Model components: Encoder, Bahdanau Attention, Decoder

In [4]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units):
        super().__init__()
        self.enc_units = enc_units
        self.embedding = layers.Embedding(vocab_size, embedding_dim)
        # Using GRU; you can use LSTM similarly
        self.gru = layers.GRU(enc_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')

    def call(self, x, hidden):
        x = self.embedding(x)  # (batch, seq_len, emb)
        output, state = self.gru(x, initial_state=hidden)  # output: (batch, seq, units)
        return output, state

    def initialize_hidden_state(self, batch_size):
        return tf.zeros((batch_size, self.enc_units))

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = layers.Dense(units)
        self.W2 = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, enc_output, dec_hidden):
        # enc_output: (batch, enc_len, units)
        # dec_hidden: (batch, units)
        dec_hidden_with_time = tf.expand_dims(dec_hidden, 1)  # (batch, 1, units)
        score = self.V(tf.nn.tanh(self.W1(enc_output) + self.W2(dec_hidden_with_time)))
        # score: (batch, enc_len, 1)
        attention_weights = tf.nn.softmax(score, axis=1)  # (batch, enc_len, 1)
        context_vector = attention_weights * enc_output  # (batch, enc_len, units)
        context_vector = tf.reduce_sum(context_vector, axis=1)  # (batch, units)
        return context_vector, tf.squeeze(attention_weights, -1)  # weights: (batch, enc_len)

class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super().__init__()
        self.dec_units = dec_units
        self.embedding = layers.Embedding(vocab_size, embedding_dim)
        self.gru = layers.GRU(dec_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')
        self.fc = layers.Dense(vocab_size)
        self.attention = BahdanauAttention(dec_units)

    def call(self, x, hidden, enc_output):
        # x: (batch, 1) token input
        x = self.embedding(x)  # (batch, 1, emb)
        context_vector, attention_weights = self.attention(enc_output, hidden)
        # context_vector -> (batch, units); expand for concat
        context_vector = tf.expand_dims(context_vector, 1)  # (batch, 1, units)
        x = tf.concat([context_vector, x], axis=-1)  # (batch, 1, units+emb)
        output, state = self.gru(x)  # output: (batch, 1, units)
        output = tf.reshape(output, (-1, output.shape[2]))  # (batch, units)
        x = self.fc(output)  # (batch, vocab)
        return x, state, attention_weights

# 5) Hyperparameters and model instantiation

In [5]:
embedding_dim = 64
units = 128
BATCH_SIZE = X_train.shape[0]  # small toy dataset -> one batch; for larger data use tf.data
BUFFER_SIZE = 2000

encoder = Encoder(src_vocab_size, embedding_dim, units)
decoder = Decoder(tgt_vocab_size, embedding_dim, units)

optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

# Loss function mask out padding tokens (0)
def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_sum(loss_) / tf.reduce_sum(mask)

# 6) Prepare training dataset (tf.data) for batching

In [6]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, decoder_input_train, decoder_target_train))
train_dataset = train_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=False)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, decoder_input_val, decoder_target_val))
val_dataset = val_dataset.batch(BATCH_SIZE)

# 7) Training step 

In [7]:
@tf.function
def train_step(inp, targ_inp, targ_real, enc_hidden):
    loss = 0
    with tf.GradientTape() as tape:
        enc_output, enc_hidden = encoder(inp, enc_hidden)
        dec_hidden = enc_hidden
        # decoder input first token for each sample (shape (batch,1))
        dec_input = tf.expand_dims(targ_inp[:, 0], 1)
        # iterate over time steps
        for t in range(1, targ_inp.shape[1]):
            # pass through decoder
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            # targ_real[:, t-1] corresponds to actual next token
            loss += loss_function(targ_real[:, t-1], predictions)
            # teacher forcing: next input is ground truth
            dec_input = tf.expand_dims(targ_inp[:, t], 1)
    batch_loss = loss / tf.cast(targ_real.shape[1], dtype=loss.dtype)
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return batch_loss

# 8) Train loop

In [10]:

for epoch in range(1, EPOCHS + 1):
    enc_hidden = encoder.initialize_hidden_state(BATCH_SIZE)
    total_loss = 0
    for (batch, (inp, targ_inp, targ_real)) in enumerate(train_dataset):
        batch_loss = train_step(inp, targ_inp, targ_real, enc_hidden)
        total_loss += batch_loss
    if epoch % 20 == 0 or epoch == 1:
        print(f'Epoch {epoch} Loss {total_loss.numpy():.4f}')


Epoch 1 Loss 0.0031
Epoch 20 Loss 0.0028
Epoch 40 Loss 0.0025
Epoch 60 Loss 0.0023
Epoch 80 Loss 0.0021
Epoch 100 Loss 0.0019
Epoch 120 Loss 0.0018
Epoch 140 Loss 0.0016
Epoch 160 Loss 0.0015
Epoch 180 Loss 0.0014
Epoch 200 Loss 0.0013
Epoch 220 Loss 0.0012
Epoch 240 Loss 0.0011


# 9) Inference (Greedy decoding)

In [12]:
index_to_word_tgt = {v: k for k, v in tgt_tokenizer.word_index.items()}
index_to_word_src = {v: k for k, v in src_tokenizer.word_index.items()}

def evaluate(sentence, max_tgt_len=max_tgt_len):
    # preprocess source sentence
    seq = src_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_src_len, padding='post')
    inp = tf.convert_to_tensor(seq)
    enc_hidden = encoder.initialize_hidden_state(1)
    enc_out, enc_hidden = encoder(inp, enc_hidden)
    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([tgt_tokenizer.word_index['<start>']], 0)  # (1,1)
    result_tokens = []
    attention_plot = np.zeros((max_tgt_len, max_src_len))
    for t in range(max_tgt_len):
        predictions, dec_hidden, attention_weights = decoder(dec_input, dec_hidden, enc_out)
        # store attention
        attention_plot[t] = attention_weights.numpy()
        predicted_id = tf.argmax(predictions[0]).numpy()
        predicted_word = index_to_word_tgt.get(predicted_id, '<unk>')
        if predicted_word == '<end>' or predicted_word is None:
            break
        result_tokens.append(predicted_word)
        dec_input = tf.expand_dims([predicted_id], 0)
    return ' '.join(result_tokens), attention_plot[:len(result_tokens), :]

def translate(sentence):
    sent = sentence.lower().strip()
    translation, attn = evaluate(sent)
    print(f"Input: {sentence}")
    print(f"Predicted translation: {translation}")

# 10) Test translations on test set

In [13]:
print("\n--- Test translations ---")
for src, tgt in test_pairs:
    translate(src)
    print("Ground truth:", tgt)
    print()


--- Test translations ---
Input: how are you
Predicted translation: à plus tard
Ground truth: comment ça va

Input: do you speak english
Predicted translation: à plus tard
Ground truth: parlez-vous anglais

